# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- **Gymnasium cumple el papel del `Environment`** (lo que en CheeseWorld era la clase `CheeseWorld`): define estados, acciones, transiciones y recompensas. Expone la misma interfaz: `env.reset()` → estado inicial y `env.step(a)` → `(s', r, terminated, truncated, info)`.
- **El agente ya no es una clase**: quedó repartido en la **Q-table** (`Qtable`, un arreglo NumPy de estados × acciones) y en las funciones `epsilon_greedy_policy` / `greedy_policy` (cómo decide) y `train_q_learning` (cómo aprende: TD target, TD error y actualización).
- La separación conceptual es la misma: el ambiente **no aprende ni decide**; el agente **no conoce** $T$ ni $R$, solo observa experiencias $(s,a,r,s')$.


## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

- **Filas:** una por estado → `env.observation_space.n` = **16** (grid 4×4).
- **Columnas:** una por acción → `env.action_space.n` = **4** (Left, Down, Right, Up).
- **$Q[s,a]$** es la estimación del **retorno esperado descontado** si el agente está en el estado $s$, ejecuta la acción $a$ y luego sigue la mejor política: $Q(s,a)\approx \mathbb{E}\left[r+\gamma\max_{a'}Q(s',a')\right]$. No es la recompensa inmediata, sino "qué tan buena es esa acción a largo plazo".


In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- $Q=0$ **no significa que todas las acciones sean malas**: significa que el agente **todavía no sabe nada**. Es solo un valor inicial; como en FrozenLake todas las recompensas son $\geq 0$, cero es una estimación "neutral".
- Si varias acciones tienen el mismo $Q$ máximo hay un **empate**. `greedy_policy` lo resuelve **al azar** (`np.random.choice` sobre `np.flatnonzero(Q[s] == max_q)`). Si siempre tomáramos la primera (`np.argmax`), el agente con $Q=0$ siempre elegiría *Left* y podría quedar atrapado sin explorar.
- Al inicio, por tanto, el comportamiento greedy es equivalente a un random walk; la información aparece solo cuando el agente llega a la meta por primera vez y recibe $r=1$.


## 3. Ejecutar una transición


In [4]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 2
reward     = 0
next_state = 1
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(\quad,\quad,\quad,\quad)
$$

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?

**Respuesta:** en la ejecución guardada en este notebook la experiencia fue

$$
(s,a,r,s')=(0,\;2,\;0,\;1)
$$

es decir: desde la casilla inicial 0, la acción **Right** (2) llevó al estado 1 con recompensa 0 (el episodio no terminó). *(La acción es aleatoria porque $\epsilon=1$, así que al re-ejecutar la tupla puede cambiar.)*

En Value Iteration no trabajábamos con experiencias sino con el **modelo**: $T(s,a,s')$ y $R(s)$ completos. Lo que **no teníamos directamente** era el **$s'$ muestreado** (y su $r$ observado): VI no "vive" la transición, sino que suma sobre **todos** los $s'$ posibles ponderados por $T(s,a,s')$. En Q-Learning ocurre lo contrario: no conocemos $T$ ni $R$, solo observamos un $s'$ y un $r$ concretos que devuelve `env.step`.


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.

**Respuesta:** el ambiente es idéntico; lo único que cambió es la **Q-table**.

- **`Q_initial`**: todo en cero → no hay información; el agente se mueve al azar y cae en un hoyo (recompensa 0).
- **`Q_early`** (50 episodios): con $\epsilon\approx 1$ casi todo fue exploración y el agente llegó a la meta **una sola vez**. Solo una celda dejó de ser cero: $Q(14,\text{Right})=0.7$ (= $\alpha\cdot r = 0.7\cdot 1$), la transición que entra directamente a la meta. El resto sigue en 0, por lo que desde el inicio la política greedy vuelve a ser un random walk entre empates → recompensa 0.
- **`Q_trained`** (10 000 episodios): la recompensa se **propagó hacia atrás** con el factor $\gamma$. Los valores convergieron a potencias de $\gamma=0.95$ según la distancia a la meta: $1.0$ (a 1 paso), $0.95$, $0.9025$, $0.857$, $0.8145$, $0.774$, $0.735$… En el estado inicial $Q(0,\cdot)=[0.735, 0.774, 0.774, 0.735]$: *Down* y *Right* empatan porque ambos están en un camino más corto de 6 pasos ($0.95^5 \approx 0.774$). Las filas de los hoyos (5, 7, 11, 12) y de la meta (15) quedan en 0 porque son terminales. Por eso el agente sigue un camino más corto y obtiene recompensa 1.


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** **14** (fila 3, columna 2: justo a la izquierda de la meta G)

**Valores Q** (de `Q_trained`):

- Left: $0.9025$
- Down: $0.95$
- Right: $1.0$
- Up: $0.9025$

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Respuesta:** $\arg\max_a Q(14,a)=$ **Right** (acción 2), con $Q=1.0$.

**Interpretación:**

> *Right* entra directamente a la meta: $r=1$ y el estado es terminal, así que $Q=1+\gamma\cdot 0 = 1$. *Down* choca con el borde y deja al agente en 14: $Q=0+0.95\cdot V(14)=0.95$. *Left* lleva a 13 y *Up* a 10, ambos a 2 pasos de la meta: $Q=0.95\cdot 0.95=0.9025$. Los valores coinciden exactamente con la ecuación de Bellman de optimalidad del ambiente determinístico: el agente aprendió $Q^*$ sin conocer el modelo.


## 5. Evaluar la política aprendida


In [9]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> La exploración sirve para **aprender** (descubrir acciones y estados nuevos y corregir estimaciones). En la evaluación queremos **medir la calidad de la política aprendida**, $\pi(s)=\arg\max_a Q(s,a)$; si siguiéramos tomando acciones aleatorias con probabilidad $\epsilon$ estaríamos midiendo una política "con ruido" y el resultado subestimaría lo que realmente aprendió el agente. Además, la evaluación no actualiza la Q-table, así que explorar no aportaría nada. Por eso, en el ambiente determinístico, la política greedy obtiene `Mean reward = 1.000` con desviación 0: llega a la meta en el 100 % de los episodios.


## 6. Experimento: FrozenLake estocástico


In [10]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.718
Std reward : 0.450


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con `is_slippery=True` las transiciones son **estocásticas**: el agente se mueve en la dirección deseada solo con probabilidad 1/3 y se desliza hacia cada una de las dos direcciones perpendiculares con probabilidad 1/3. La misma $(s,a)$ puede llevar a distintos $s'$. Por eso, aun con la política óptima, no se llega siempre a la meta: en esta ejecución la política aprendida logra `Mean reward ≈ 0.72` (72 % de éxito, con desviación ≈ 0.45), frente a `1.000` en el caso determinístico. Además el entrenamiento es más ruidoso y necesita más episodios (20 000 y un `decay_rate` menor) y, al re-ejecutar, el resultado varía bastante entre corridas.
- **Algoritmo:** **la ecuación de Q-Learning no cambia en nada**: $Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right]$. Lo que cambia es la interpretación: ahora cada $s'$ es una **muestra** de $T(s,a,\cdot)$, y la actualización hace un promedio móvil de muchas muestras para aproximar el valor esperado $\sum_{s'}T(s,a,s')[\dots]$. Esto hace importante a $\alpha$: con $\alpha=0.7$ cada muestra pesa mucho y $Q$ oscila; un $\alpha$ más pequeño promediaría mejor la aleatoriedad del ambiente.


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> La estimación del **retorno esperado descontado** al ejecutar la acción $a$ en el estado $s$ y luego actuar de forma óptima (greedy). Es decir, qué tan buena es la pareja estado–acción a largo plazo, no solo su recompensa inmediata.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la **propia Q-table**, evaluada en el estado siguiente $s'$ que devolvió el ambiente. Es la mejor estimación actual del valor futuro desde $s'$ (*bootstrapping*): se usa una estimación para actualizar otra. Si $s'$ es terminal se toma 0. Junto con $r$ forma el **TD target** $r+\gamma\max_{a'}Q(s',a')$.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar **exploración y explotación**. Si el agente solo explotara, se quedaría con las primeras acciones que parecen buenas (o con empates en 0) y nunca descubriría la meta ni caminos mejores. Con probabilidad $\epsilon$ prueba acciones aleatorias y con $1-\epsilon$ usa lo aprendido. Además, $\epsilon$ decae: mucha exploración al inicio y más explotación al final.

**4. ¿Por qué Q-Learning es model-free?**

> Porque **nunca usa ni estima** $T(s,a,s')$ ni $R$. Aprende directamente de las experiencias $(s,a,r,s')$ que devuelve `env.step`, a diferencia de Value/Policy Iteration, que necesitan el modelo completo para sumar sobre todos los $s'$. El agente aprende "qué acción conviene" sin saber cómo funciona el mundo por dentro.
